In [9]:
import fiona
import pandas as pd
from shapely.geometry import shape
from shapely.wkt import dumps as wkt_dumps
from shapely.ops import transform  # <-- Fix: import transform directly!
import os
import pyproj

def shapefile_to_csv(shapefile_path, output_csv_path):
    with fiona.open(shapefile_path, 'r') as src:
        features = list(src)
        crs = src.crs
        crs_obj = pyproj.CRS(crs)
        is_projected = crs_obj.is_projected

        records = []
        for feat in features:
            props = dict(feat['properties'])
            geom = shape(feat['geometry'])

            # Project geometry to WGS84 for lat/lon output if needed
            if is_projected:
                # Project from projected CRS to WGS84 (EPSG:4326)
                project = pyproj.Transformer.from_crs(crs_obj, pyproj.CRS("EPSG:4326"), always_xy=True).transform
                geom_latlon = transform(project, geom)
            else:
                geom_latlon = geom

            # Basic geometric properties
            props['geometry_type'] = geom.geom_type
            props['wkt'] = wkt_dumps(geom)
            props['area'] = geom.area
            props['bbox_xmin'], props['bbox_ymin'], props['bbox_xmax'], props['bbox_ymax'] = geom.bounds

            # Centroid (in WGS84)
            props['centroid_lon'] = geom_latlon.centroid.x
            props['centroid_lat'] = geom_latlon.centroid.y

            # Optionally: all exterior coordinates as list
            if geom.geom_type == "Polygon":
                props['coords'] = list(geom.exterior.coords)
            elif geom.geom_type == "MultiPolygon":
                props['coords'] = [list(p.exterior.coords) for p in geom.geoms]
            else:
                props['coords'] = list(geom.coords)

            records.append(props)

    # DataFrame and CSV
    df = pd.DataFrame(records)
    df.to_csv(output_csv_path, index=False)
    print(f"✅ CSV saved to: {output_csv_path}")


# Example usage:
shapefile_to_csv(
    shapefile_path="CSV_Yolo_test/final_rect_merged_sup/bbox_shapes_filtered.shp",
    output_csv_path="CSV_Yolo_test/bbox_shapes_filtered.csv"
)



In [10]:
# Example usage:
shapefile_to_csv(
    shapefile_path="CSV_Yolo_test/final_rect_merged_sup/bbox_shapes_filtered.shp",
    output_csv_path="CSV_Yolo_test/bbox_shapes_filtered.csv"
)


✅ CSV saved to: CSV_Yolo_test/bbox_shapes_filtered.csv


In [25]:
import os
import fiona
import pandas as pd
from shapely.geometry import shape
from shapely.wkt import dumps as wkt_dumps
from shapely.ops import transform
import pyproj

def shapefile_to_projected_csv(
    shapefile_path, 
    output_csv_path,
    target_epsg="26916"  # Set your target EPSG code here (e.g., "26916" for UTM 16N)
):
    target_crs = pyproj.CRS.from_epsg(target_epsg)

    with fiona.open(shapefile_path, 'r') as src:
        src_crs = src.crs
        src_crs_obj = pyproj.CRS(src_crs)
        need_reproject = src_crs_obj != target_crs
        print(f"Source CRS: {src_crs_obj}")
        print(f"Target CRS: {target_crs}")

        records = []
        for feat in src:
            props = dict(feat['properties'])
            geom = shape(feat['geometry'])

            # Reproject if necessary
            if need_reproject:
                project = pyproj.Transformer.from_crs(src_crs_obj, target_crs, always_xy=True).transform
                geom_proj = transform(project, geom)
            else:
                geom_proj = geom

            props['geometry_type'] = geom_proj.geom_type
            props['wkt'] = wkt_dumps(geom_proj)
            props['area'] = geom_proj.area
            props['bbox_xmin'], props['bbox_ymin'], props['bbox_xmax'], props['bbox_ymax'] = geom_proj.bounds

            # Centroid in projected CRS
            props['centroid_x'] = geom_proj.centroid.x
            props['centroid_y'] = geom_proj.centroid.y

            # Optionally: all exterior coordinates as list
            if geom_proj.geom_type == "Polygon":
                props['coords'] = list(geom_proj.exterior.coords)
            elif geom_proj.geom_type == "MultiPolygon":
                props['coords'] = [list(p.exterior.coords) for p in geom_proj.geoms]
            else:
                props['coords'] = list(geom_proj.coords)

            records.append(props)

    df = pd.DataFrame(records)
    df.to_csv(output_csv_path, index=False)
    print(f"✅ Projected CSV saved to: {output_csv_path}")

# Example usage:
shapefile_to_projected_csv(
    shapefile_path="CSV_Yolo_test/final_rect_merged_sup/bbox_shapes_filtered.shp",
    output_csv_path="CSV_Yolo_test/bbox_shapes_filtered_projected.csv",
    target_epsg="26916"   # UTM 16N
)

Source CRS: PROJCS["NAD83 / UTM zone 16N",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-87],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","26916"]]
Target CRS: EPSG:26916
✅ Projected CSV saved to: CSV_Yolo_test/bbox_shapes_filtered_projected.csv


In [29]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from tqdm import tqdm
import torch
from segment_anything import sam_model_registry, SamPredictor

# SAM Setup
CHECKPOINT_PATH = "weights/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
sam_predictor = SamPredictor(sam)

def proj_bbox_to_pixel(bbox_proj, transform):
    x1, y1, x2, y2 = bbox_proj
    row1, col1 = rowcol(transform, x1, y1)
    row2, col2 = rowcol(transform, x2, y2)
    px1, py1 = col1, row1
    px2, py2 = col2, row2
    x1, x2 = sorted([px1, px2])
    y1, y2 = sorted([py1, py2])
    return x1, y1, x2, y2

def run_sam_on_bbox_csv(
    csv_path, geotiff_dir, output_dir,
    class_label_col="class", conf_col="confidence",
    tile_col="tile_name",
    xmin_col="bbox_xmin", ymin_col="bbox_ymin",
    xmax_col="bbox_xmax", ymax_col="bbox_ymax"
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    tiff_files = {os.path.splitext(os.path.basename(f))[0]: f for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tile_base, tiff_file in tqdm(tiff_files.items()):
        objects = df[df[tile_col] == (tile_base + ".tif")].to_dict(orient='records')
        if not objects:
            continue

        geotiff_path = os.path.join(geotiff_dir, tiff_file)
        with rasterio.open(geotiff_path) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        for obj in objects:
            try:
                bbox_proj = [
                    float(obj[xmin_col]), float(obj[ymin_col]),
                    float(obj[xmax_col]), float(obj[ymax_col])
                ]
                x1, y1, x2, y2 = proj_bbox_to_pixel(bbox_proj, transform)
                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                if x2 <= x1 or y2 <= y1:
                    continue

                roi_rgb = overlay[y1:y2, x1:x2]
                if roi_rgb.size == 0 or roi_rgb.shape[0] < 2 or roi_rgb.shape[1] < 2:
                    continue

                sam_predictor.set_image(roi_rgb)
                box = np.array([0, 0, x2-x1, y2-y1])
                masks, scores, logits = sam_predictor.predict(box=box, multimask_output=True)
                if masks is None or len(masks) == 0:
                    continue
                mask = masks[np.argmax(scores)]

                mask_full = np.zeros(overlay.shape[:2], dtype=np.uint8)
                mask_resized = cv2.resize(mask.astype(np.uint8), (x2-x1, y2-y1), interpolation=cv2.INTER_NEAREST)
                mask_full[y1:y2, x1:x2] = mask_resized

                mask_img = np.zeros_like(overlay)
                mask_img[mask_full > 0] = [0, 0, 255]
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.4, 0)

                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label = f"Cls:{obj.get(class_label_col, '')} ({float(obj.get(conf_col, 0.0)):.2f})"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 1.0
                thickness = 2
                text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
                text_origin = (x1, max(y1 - 5, text_size[1]))
                cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness+2)
                cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)
            except Exception:
                continue

        # Convert RGB (default) to BGR for OpenCV before saving as JPG
        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, tile_base + "_sam_annotated.jpg")
        cv2.imwrite(out_path, overlay_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])

# ---------- Usage ----------
run_sam_on_bbox_csv(
    csv_path="CSV_Yolo_test/bbox_shapes_filtered_projected.csv",
    geotiff_dir="test_sample",
    output_dir="annotated_geotiff_jpg"
)


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [18:35<00:00, 111.56s/it]
